In [2]:
from astroquery.sdss import SDSS
from astroquery.ipac.irsa import Irsa
from astroquery.vizier import Vizier
from astropy import coordinates as coord
from astropy import units as u
from astroquery.ipac.ned import Ned
import numpy as np
import pandas as pd

In [49]:
# Coordenadas de ejemplo
ra, dec = 1.7376141,0.8571812 # en grados
pos = coord.SkyCoord(ra, dec, unit=(u.deg, u.deg), frame='icrs')

# ---- 1. SDSS (u,g,r,i,z) ----
sdss_data = SDSS.query_region(
    pos,
    radius=0.15 * u.arcsec,   # <--- este es el argumento que faltaba
    spectro=False,
    photoobj_fields=[
        'ra','dec',
        'modelMag_u','modelMagErr_u',
        'modelMag_g','modelMagErr_g',
        'modelMag_r','modelMagErr_r',
        'modelMag_i','modelMagErr_i',
        'modelMag_z','modelMagErr_z'
    ]
)

sdss_data

objID,ra,dec,modelMag_u,modelMagErr_u,modelMag_g,modelMagErr_g,modelMag_r,modelMagErr_r,modelMag_i,modelMagErr_i,modelMag_z,modelMagErr_z
uint64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
1237645879019765919,1.73762334596665,0.85717687320632,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0
1237645879019831364,1.73757747583687,0.857196920707493,19.13904,0.02542134,18.5538,0.008198288,18.76767,0.01176654,18.21628,0.01195736,18.57623,0.05468069


In [50]:
# Coordenadas de ejemplo
ra, dec = 1.7376141,0.8571812 # en grados
pos = coord.SkyCoord(ra, dec, unit=(u.deg, u.deg), frame='icrs')

# ---- 4. WISE (W1–W4) ----
wise = Irsa.query_region(pos, 
                         catalog='allwise_p3as_psd', 
                         spatial='Cone', 
                         radius=0.15*u.arcsec)
wise['w1mpro','w1sigmpro','w2mpro','w2sigmpro','w3mpro','w3sigmpro','w4mpro','w4sigmpro']


w1mpro,w1sigmpro,w2mpro,w2sigmpro,w3mpro,w3sigmpro,w4mpro,w4sigmpro
mag,mag,mag,mag,mag,mag,mag,mag
float32,float32,float32,float32,float32,float32,float32,float32
16.286,0.074,15.521,0.121,10.896,0.132,7.271,0.123


In [114]:
result_table = Ned.get_table("UM 199", table='photometry')


result_table[result_table['Observed Passband']=='W1 (WISE)']

No.,Observed Passband,Photometry Measurement,Uncertainty,Units,Frequency,Flux Density,Upper limit of uncertainty,Lower limit of uncertainty,Upper limit of Flux Density,Lower limit of Flux Density,NED Uncertainty,NED Units,Refcode,Significance,Published frequency,Frequency Mode,Coordinates Targeted,Spatial Mode,Qualifiers,Comments
,,,,,Hz,Jy,,,,,,,,,,,,,,
int32,object,float64,object,object,float64,float64,float64,float64,float64,float64,object,object,object,object,object,object,object,object,object,object
45,W1 (WISE),16.286,+/-0.074,mag,89400000000000.0,9.47e-05,6.45e-06,6.45e-06,--,--,+/-6.45E-06,Jy,2013wise.rept....1C,uncertainty,3.3526 microns,Broad-band measurement,,From fitting to map,Profile-fit,From new raw data
46,W1 (WISE),15.944,+/-0.121,mag,89400000000000.0,0.00013,1.45e-05,1.45e-05,--,--,+/-1.45E-05,Jy,2013wise.rept....1C,uncertainty,3.3526 microns,Broad-band measurement,,Flux in fixed aperture,"r=8.25"" COG-corrected",From new raw data; Uncorrected for known sources in beam
47,W1 (WISE),14.442,+/-0.062,mag,89400000000000.0,0.000518,2.96e-05,2.96e-05,--,--,+/-2.96E-05,Jy,2013wise.rept....1C,uncertainty,3.3526 microns,Broad-band measurement,,Flux in fixed aperture,"r=22.0"" aperture",From new raw data; Uncorrected for known sources in beam


In [ ]:
np.unique(result_table['Observed Passband'])

FUV (GALEX) AB
H{beta} line
NUV (GALEX) AB
W1 (WISE)
W2 (WISE)
W3 (WISE)
W4 (WISE)
g (SDSS CModel) AB
g (SDSS Model) AB
g (SDSS PSF) AB
g (SDSS Petrosian)AB


In [124]:
import os
HOME = os.path.expanduser("~") + '/gdrive/DataHII/'

def magAB_to_flux(mag, mag_err):
    f = 10**(-0.4 * (mag - 8.90))
    ferr = f * np.log(10)/2.5 * mag_err
    return f, ferr

WISE_ZP = {
    'W1': 309.540,
    'W2': 171.787,
    'W3': 31.674,
    'W4': 8.363
}

def vega_to_jy(mag, mag_err, band):
    """Convierte magnitudes Vega de WISE a flux (Jy) con error."""
    f0 = WISE_ZP[band.upper()]
    f = f0 * 10**(-0.4 * mag)
    ferr = f * np.log(10)/2.5 * mag_err
    return f, ferr

def GALEX_data(table,filter):
    TABLE = table[(table['Observed Passband']==filter) & (table['Qualifiers']=='Kron flux in elliptical aperture')]

    errors = np.array(TABLE['Uncertainty'])

    Err_min = np.min(errors)

    CHOSEN = TABLE[TABLE['Uncertainty']==Err_min]


    flx = CHOSEN['Photometry Measurement'][0]
    flx_err = float(CHOSEN['Uncertainty'][0].replace("+/-",""))

    fx_mJy, fxerr_mJy = magAB_to_flux(flx, flx_err)

    #return flx, flx_err
    #return fx_mJy, fxerr_mJy
    return fx_mJy * 1e3 , fxerr_mJy * 1e3

def SDSS_data(table,filter):
    TABLE = table[(table['Observed Passband']==filter)]

    Ref = np.array(TABLE['Refcode'])

    Older_ref = np.max(Ref)

    CHOSEN = TABLE[TABLE['Refcode']==Older_ref]


    flx = CHOSEN['Photometry Measurement'][0]
    flx_err = float(CHOSEN['Uncertainty'][0].replace("+/-",""))

    fx_mJy, fxerr_mJy = magAB_to_flux(flx, flx_err)

    #return flx, flx_err
    #return fx_mJy, fxerr_mJy
    return fx_mJy * 1e3 , fxerr_mJy * 1e3

def WISE_data(table,filter):

    band = filter[0:2]

    TABLE = table[(table['Observed Passband']==filter) & (table['Qualifiers']=='Profile-fit')]

    CHOSEN = TABLE


    flx = CHOSEN['Photometry Measurement'][0]
    flx_err = float(CHOSEN['Uncertainty'][0].replace("+/-",""))

    fx_mJy, fxerr_mJy = vega_to_jy(flx, flx_err,band)

    #return flx, flx_err
    #return fx_mJy, fxerr_mJy
    return fx_mJy * 1e3 , fxerr_mJy * 1e3

In [139]:
# Sacando los datos para CIGALE

DF = pd.read_csv(HOME+'/SPS/HIIGs_aladin.csv')

head = "# id redshift galex.FUV galex.FUV_err galex.NUV galex.NUV_err "
head = head + "sdss.up sdss.up_err sdss.gp sdss.gp_err sdss.rp sdss.rp_err sdss.ip sdss.ip_err sdss.zp sdss.zp_err "
head = head + "WISE1 WISE1_err WISE2 WISE2_err WISE3 WISE3_err WISE4 WISE4_err"

with open("cigale_HIIG_sample.txt", 'w') as file:
    file.write(head)

    for n in range(len(DF)):

        name = DF['Name'][n]

        filters = ['FUV (GALEX) AB', 'NUV (GALEX) AB','u (SDSS Model) AB','g (SDSS Model) AB','r (SDSS Model) AB','i (SDSS Model) AB','z (SDSS Model) AB'
                  'W1 (WISE)','W2 (WISE)','W3 (WISE)','W4 (WISE)']
        result_table = Ned.get_table(name, table='photometry')
        ned_filters = np.unique(result_table['Observed Passband'])

        fuv,nuv,u,g,r,i,z,W1,W2,W3,W4 = 0,0,0,0,0,0,0,0,0,0,0
        e1,e2,e3,e4,e5,e6,e7,e8,e9,e10,e11 = 0,0,0,0,0,0,0,0,0,0,0

        

        #if 'FUV (GALEX) AB' in filters:



In [133]:
#Numero de galaxias con datos completos

DF = pd.read_csv(HOME+'/SPS/HIIGs_aladin.csv')

HIIG_names = np.array(DF['Name'])

print('Name | FUV | NUV | u | g | r | i | z | W1 | W2 | W3 | W4 | ALL |')

sep = ' | '

name_id = 0

Found = 0

for name in HIIG_names:
    result_table = Ned.get_table(name, table='photometry')
    filters = np.unique(result_table['Observed Passband'])

    c = 0 

    tab = str(name_id) + '_id' + sep

    #FUV
    if 'FUV (GALEX) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #NUV
    if 'NUV (GALEX) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #u
    if 'u (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #g
    if 'g (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #r
    if 'r (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #i
    if 'i (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #z
    if 'z (SDSS Model) AB' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W1
    if 'W1 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W2
    if 'W2 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W3
    if 'W3 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    #W4
    if 'W4 (WISE)' in filters:
        tab += str(1) + sep
        c += 1
    else:
        tab += str(0) + sep
    


    if c == 11:
        tab += '  All found' + sep
        Found += 1
    else:
        tab += str(0) + sep


    #print(tab)

    name_id += 1

print(f'\n Found complete:{Found}')    
    


Name | FUV | NUV | u | g | r | i | z | W1 | W2 | W3 | W4 | ALL |

 Found complete:80


In [74]:
vega_to_jy(14.442,0.062,'W1')

(0.0005175076549690697, 2.955181421368373e-05)

In [47]:
Irsa.ROW_LIMIT = 10
galex = Irsa.query_region(pos, catalog='galex_emphot_v3', spatial='Cone', radius=15*u.arcmin)
galex

xnumber,ra,dec,pointing_nuv,pointing_fuv,x_nuv,x_fuv,y_nuv,y_fuv,bkg_nuv,bkg_fuv,flux_nuv,flux_fuv,fluxerr_nuv,fluxerr_fuv,mag_nuv,mag_fuv,merr_nuv,merr_fuv,mag_prior,mask_nuv,mask_fuv,class_star,x,y,z,spt_ind,cntr
,deg,deg,,,pixel,pixel,pixel,pixel,count/s,count/s,count/s,count/s,count/s,count/s,,,,,,,,,,,,,
int64,float64,float64,int64,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,int64,float32,float64,float64,float64,int64,int64


In [43]:
Irsa.list_catalogs(filter='galex')

{'cosmos.cosmos_galex': 'COSMOS Galex Image Metadata',
 'galex_emphot_v3': 'GALEX/COSMOS Prior-based Photometry Catalog June 2008',
 'spitzer.lvl_galex': 'Spitzer LVL GALEX image Metadata'}